In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [4]:
import pandas as pd
from pathlib import Path

# Path from notebooks/ → data/raw/
DATA_RAW = Path("../data/raw")

In [6]:
import os
from pathlib import Path

print("Current working directory:")
print(os.getcwd())

print("\nFiles/folders here:")
for item in Path.cwd().iterdir():
    print(item.name)

Current working directory:
C:\Users\Mitalika\OneDrive\Desktop\enterprise_hr_ai\notebooks

Files/folders here:
.ipynb_checkpoints
01_data_audit.ipynb
02_data_cleaning.ipynb.ipynb


In [7]:
from pathlib import Path

# Search for the HR dataset from the current location
project_root = None

for path in [Path.cwd(), *Path.cwd().parents]:
    if (path / "data" / "raw").exists():
        project_root = path
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find the enterprise_hr_ai/data/raw folder."
    )

DATA_RAW = project_root / "data" / "raw"

print("Project root:", project_root)
print("Raw data folder:", DATA_RAW)

print("\nFiles found:")
for file in DATA_RAW.iterdir():
    print(" -", file.name)

Project root: C:\Users\Mitalika\OneDrive\Desktop\enterprise_hr_ai
Raw data folder: C:\Users\Mitalika\OneDrive\Desktop\enterprise_hr_ai\data\raw

Files found:
 - Cleaned_HR_Data_Analysis.csv
 - Employee_Performance_Dataset.csv
 - employee_performance_pro.csv
 - essential_skills.csv
 - Messy_HR_Dataset_Detailed.csv
 - Occupation Data.xlsx
 - Software Skills.xlsx
 - WA_Fn-UseC_-HR-Employee-Attrition.csv


In [8]:
import pandas as pd

# Load datasets
clean_hr = pd.read_csv(
    DATA_RAW / "Cleaned_HR_Data_Analysis.csv"
)

messy_hr = pd.read_csv(
    DATA_RAW / "Messy_HR_Dataset_Detailed.csv"
)

attrition_hr = pd.read_csv(
    DATA_RAW / "WA_Fn-UseC_-HR-Employee-Attrition.csv"
)

onet_hr = pd.read_excel(
    DATA_RAW / "Occupation Data.xlsx"
)

print("DATASETS LOADED")
print("=" * 80)

print("clean_hr     :", clean_hr.shape)
print("messy_hr     :", messy_hr.shape)
print("attrition_hr :", attrition_hr.shape)
print("onet_hr      :", onet_hr.shape)

DATASETS LOADED
clean_hr     : (2845, 28)
messy_hr     : (3150, 39)
attrition_hr : (1470, 35)
onet_hr      : (1016, 3)


In [9]:
# ============================================================
# STEP 3: DATE STANDARDIZATION
# ============================================================

clean_hr = clean_hr.copy()

date_columns = [
    "StartDate",
    "DOB",
    "Survey Date",
    "Training Date"
]

for col in date_columns:
    clean_hr[col] = pd.to_datetime(
        clean_hr[col],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

print("DATE CONVERSION COMPLETE")
print("=" * 80)

for col in date_columns:
    print(
        f"{col:<20} "
        f"dtype={clean_hr[col].dtype} | "
        f"missing={clean_hr[col].isna().sum()}"
    )

DATE CONVERSION COMPLETE
StartDate            dtype=datetime64[ns] | missing=0
DOB                  dtype=datetime64[ns] | missing=0
Survey Date          dtype=datetime64[ns] | missing=0
Training Date        dtype=datetime64[ns] | missing=0


In [10]:
print("\nDATE RANGES")
print("=" * 80)

for col in date_columns:
    print(
        f"{col:<20} "
        f"Min: {clean_hr[col].min().date()} | "
        f"Max: {clean_hr[col].max().date()}"
    )


DATE RANGES
StartDate            Min: 2018-08-07 | Max: 2023-08-06
DOB                  Min: 1941-08-14 | Max: 2001-07-09
Survey Date          Min: 2022-08-05 | Max: 2023-08-05
Training Date        Min: 2022-08-05 | Max: 2023-08-05


In [11]:
print("\nDATE PARSING CHECK")
print("=" * 80)

for col in date_columns:
    invalid = clean_hr[col].isna().sum()
    print(f"{col:<20}: {invalid} invalid dates")


DATE PARSING CHECK
StartDate           : 0 invalid dates
DOB                 : 0 invalid dates
Survey Date         : 0 invalid dates
Training Date       : 0 invalid dates


In [12]:
# ============================================================
# STEP 5: CREATE ACCURATE AGE AT JOINING
# ============================================================

clean_hr["AgeCalculated"] = (
    clean_hr["StartDate"].dt.year
    - clean_hr["DOB"].dt.year
    - (
        (clean_hr["StartDate"].dt.month < clean_hr["DOB"].dt.month)
        |
        (
            (clean_hr["StartDate"].dt.month == clean_hr["DOB"].dt.month)
            &
            (clean_hr["StartDate"].dt.day < clean_hr["DOB"].dt.day)
        )
    ).astype(int)
)

clean_hr["AgeDifference"] = (
    clean_hr["Age"] - clean_hr["AgeCalculated"]
)

print("AGE STANDARDIZATION COMPLETE")
print("=" * 80)

print(clean_hr[
    ["Employee ID", "DOB", "StartDate", "Age", "AgeCalculated", "AgeDifference"]
].head(10))

AGE STANDARDIZATION COMPLETE
   Employee ID        DOB  StartDate  Age  AgeCalculated  AgeDifference
0         3427 1969-10-07 2019-09-20   50             49              1
1         3428 1965-08-30 2023-02-11   58             57              1
2         3429 1991-10-06 2018-12-10   27             27              0
3         3430 1998-04-04 2021-06-21   23             23              0
4         3431 1969-08-29 2019-06-29   50             49              1
5         3432 1949-04-03 2020-01-17   71             70              1
6         3433 1942-07-01 2022-04-06   80             79              1
7         3434 1957-03-07 2020-11-06   63             63              0
8         3435 1974-05-15 2018-08-18   44             44              0
9         3436 1949-11-11 2022-01-21   73             72              1


In [13]:
print("\nAGE DIFFERENCE")
print("=" * 80)

print(clean_hr["AgeDifference"].value_counts().sort_index())


AGE DIFFERENCE
AgeDifference
0    1463
1    1382
Name: count, dtype: int64


In [14]:
# ============================================================
# STEP 6: STANDARDIZE CATEGORICAL TEXT
# ============================================================

categorical_columns = clean_hr.select_dtypes(
    include=["object", "string"]
).columns

for col in categorical_columns:
    clean_hr[col] = (
        clean_hr[col]
        .astype("string")
        .str.strip()
    )

print("CATEGORICAL TEXT STANDARDIZATION COMPLETE")
print("=" * 80)

print("Categorical columns processed:")
for col in categorical_columns:
    print(f"- {col}")

CATEGORICAL TEXT STANDARDIZATION COMPLETE
Categorical columns processed:
- Title
- BusinessUnit
- EmployeeStatus
- EmployeeType
- PayZone
- EmployeeClassificationType
- DepartmentType
- Division
- State
- GenderCode
- RaceDesc
- MaritalDesc
- Performance Score
- Training Program Name
- Training Type
- Training Outcome


In [15]:
print("\nNULL CHECK AFTER TEXT STANDARDIZATION")
print("=" * 80)

print(
    clean_hr[categorical_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)


NULL CHECK AFTER TEXT STANDARDIZATION
Title                         0
BusinessUnit                  0
EmployeeStatus                0
EmployeeType                  0
PayZone                       0
EmployeeClassificationType    0
DepartmentType                0
Division                      0
State                         0
GenderCode                    0
RaceDesc                      0
MaritalDesc                   0
Performance Score             0
Training Program Name         0
Training Type                 0
Training Outcome              0
dtype: int64


In [16]:
# ============================================================
# STEP 3: DATE STANDARDIZATION
# ============================================================

date_columns = [
    "StartDate",
    "DOB",
    "Survey Date",
    "Training Date"
]

for col in date_columns:
    clean_hr[col] = pd.to_datetime(
        clean_hr[col],
        errors="coerce"
    )

for col in ["StartDate", "ExitDate", "DOB", "Survey Date", "Training Date"]:
    messy_hr[col] = pd.to_datetime(
        messy_hr[col],
        errors="coerce"
    )

print("=" * 80)
print("DATE STANDARDIZATION")
print("=" * 80)

for col in date_columns:
    print(
        f"{col:20} | "
        f"dtype: {clean_hr[col].dtype} | "
        f"nulls: {clean_hr[col].isna().sum()}"
    )

C:\Users\Mitalika\AppData\Local\Temp\ipykernel_15680\3857444830.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  messy_hr[col] = pd.to_datetime(


DATE STANDARDIZATION
StartDate            | dtype: datetime64[ns] | nulls: 0
DOB                  | dtype: datetime64[ns] | nulls: 0
Survey Date          | dtype: datetime64[ns] | nulls: 0
Training Date        | dtype: datetime64[ns] | nulls: 0


C:\Users\Mitalika\AppData\Local\Temp\ipykernel_15680\3857444830.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  messy_hr[col] = pd.to_datetime(
C:\Users\Mitalika\AppData\Local\Temp\ipykernel_15680\3857444830.py:19: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  messy_hr[col] = pd.to_datetime(
C:\Users\Mitalika\AppData\Local\Temp\ipykernel_15680\3857444830.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  messy_hr[col] = pd.to_datetime(


In [17]:
# ============================================================
# STEP 3: DATE VALIDATION
# ============================================================

print("=" * 80)
print("DATE VALIDATION")
print("=" * 80)

date_cols = [
    "StartDate",
    "DOB",
    "Survey Date",
    "Training Date"
]

for col in date_cols:
    print(f"\n{col}")
    print("-" * 40)
    print("Min :", clean_hr[col].min())
    print("Max :", clean_hr[col].max())
    print("Null:", clean_hr[col].isna().sum())

DATE VALIDATION

StartDate
----------------------------------------
Min : 2018-08-07 00:00:00
Max : 2023-08-06 00:00:00
Null: 0

DOB
----------------------------------------
Min : 1941-08-14 00:00:00
Max : 2001-07-09 00:00:00
Null: 0

Survey Date
----------------------------------------
Min : 2022-08-05 00:00:00
Max : 2023-08-05 00:00:00
Null: 0

Training Date
----------------------------------------
Min : 2022-08-05 00:00:00
Max : 2023-08-05 00:00:00
Null: 0


In [18]:
print("\n" + "=" * 80)
print("MESSY HR DATE VALIDATION")
print("=" * 80)

for col in date_cols:
    print(f"\n{col}")
    print("-" * 40)
    print("Min :", messy_hr[col].min())
    print("Max :", messy_hr[col].max())
    print("Null:", messy_hr[col].isna().sum())


MESSY HR DATE VALIDATION

StartDate
----------------------------------------
Min : 2018-08-07 00:00:00
Max : 2023-08-06 00:00:00
Null: 0

DOB
----------------------------------------
Min : 1941-02-10 00:00:00
Max : 2001-11-04 00:00:00
Null: 1908

Survey Date
----------------------------------------
Min : 2022-08-05 00:00:00
Max : 2023-08-05 00:00:00
Null: 0

Training Date
----------------------------------------
Min : 2022-08-05 00:00:00
Max : 2023-08-05 00:00:00
Null: 0


In [19]:
# ============================================================
# STEP 4: CREATE DATA QUALITY FLAGS
# ============================================================

print("=" * 80)
print("CREATING DATA QUALITY FLAGS")
print("=" * 80)

# Training happened before employee started
clean_hr["TrainingBeforeStart"] = (
    clean_hr["Training Date"] < clean_hr["StartDate"]
)

# Survey happened before employee started
clean_hr["SurveyBeforeStart"] = (
    clean_hr["Survey Date"] < clean_hr["StartDate"]
)

# Age consistency
calculated_age = (
    clean_hr["StartDate"].dt.year
    - clean_hr["DOB"].dt.year
    - (
        (clean_hr["StartDate"].dt.month < clean_hr["DOB"].dt.month)
        |
        (
            (clean_hr["StartDate"].dt.month == clean_hr["DOB"].dt.month)
            &
            (clean_hr["StartDate"].dt.day < clean_hr["DOB"].dt.day)
        )
    )
)

clean_hr["CalculatedAge"] = calculated_age

clean_hr["AgeDifference"] = (
    clean_hr["Age"] - clean_hr["CalculatedAge"]
)

clean_hr["AgeMismatch"] = (
    clean_hr["AgeDifference"] != 0
)


# Summary
print("\nTraining before StartDate:",
      clean_hr["TrainingBeforeStart"].sum())

print("Survey before StartDate:",
      clean_hr["SurveyBeforeStart"].sum())

print("Age mismatches:",
      clean_hr["AgeMismatch"].sum())

print("\nAge difference distribution:")
print(clean_hr["AgeDifference"].value_counts().sort_index())

CREATING DATA QUALITY FLAGS

Training before StartDate: 273
Survey before StartDate: 272
Age mismatches: 1382

Age difference distribution:
AgeDifference
0    1463
1    1382
Name: count, dtype: int64


In [20]:
# ============================================================
# STEP 5: INVESTIGATE MISSING DOB
# ============================================================

print("=" * 80)
print("MISSING DOB INVESTIGATION")
print("=" * 80)

missing_dob = messy_hr["DOB"].isna()

print("Messy HR missing DOB:", missing_dob.sum())

missing_dob_ids = set(
    messy_hr.loc[missing_dob, "Employee ID"]
)

clean_ids = set(clean_hr["Employee ID"])

recoverable_ids = missing_dob_ids & clean_ids

print("Missing DOB employees:", len(missing_dob_ids))
print("Recoverable from Clean HR:", len(recoverable_ids))
print("Not recoverable:", len(missing_dob_ids - clean_ids))

MISSING DOB INVESTIGATION
Messy HR missing DOB: 1908
Missing DOB employees: 1823
Recoverable from Clean HR: 1727
Not recoverable: 96


In [21]:
# ============================================================
# STEP 6: RECOVER DOB FROM CLEAN HR
# ============================================================

print("=" * 80)
print("DOB RECOVERY FROM CLEAN HR")
print("=" * 80)

# Create Employee ID → DOB lookup
dob_lookup = (
    clean_hr[["Employee ID", "DOB"]]
    .drop_duplicates("Employee ID")
    .set_index("Employee ID")["DOB"]
)

# Count missing DOB before recovery
missing_before = messy_hr["DOB"].isna().sum()

# Fill ONLY missing DOB values using Employee ID
messy_hr["DOB"] = messy_hr["DOB"].fillna(
    messy_hr["Employee ID"].map(dob_lookup)
)

# Count missing DOB after recovery
missing_after = messy_hr["DOB"].isna().sum()

print("Missing DOB before recovery :", missing_before)
print("Missing DOB after recovery  :", missing_after)
print("DOB values recovered        :", missing_before - missing_after)

DOB RECOVERY FROM CLEAN HR
Missing DOB before recovery : 1908
Missing DOB after recovery  : 98
DOB values recovered        : 1810


In [22]:
# ============================================================
# STEP 7: FLAG UNRECOVERABLE DOB
# ============================================================

messy_hr["DOBMissing"] = messy_hr["DOB"].isna()

print("\n" + "=" * 80)
print("DOB QUALITY STATUS")
print("=" * 80)

print("DOB missing after recovery:",
      messy_hr["DOBMissing"].sum())

print("DOB available:",
      (~messy_hr["DOBMissing"]).sum())


DOB QUALITY STATUS
DOB missing after recovery: 98
DOB available: 3052


In [23]:
# ============================================================
# STEP 8: DUPLICATE EMPLOYEE ID INVESTIGATION
# ============================================================

print("=" * 80)
print("DUPLICATE EMPLOYEE ID INVESTIGATION")
print("=" * 80)

# Find duplicated Employee IDs
duplicate_ids = (
    messy_hr.loc[
        messy_hr["Employee ID"].duplicated(keep=False),
        "Employee ID"
    ]
    .unique()
)

print("Duplicate Employee IDs:", len(duplicate_ids))

# Total rows belonging to duplicate IDs
duplicate_rows = messy_hr[
    messy_hr["Employee ID"].isin(duplicate_ids)
].copy()

print("Rows belonging to duplicate IDs:", len(duplicate_rows))

# How many rows per duplicated employee?
duplicate_counts = (
    duplicate_rows["Employee ID"]
    .value_counts()
)

print("\nRows per duplicated employee:")
print(duplicate_counts.value_counts().sort_index())

print("\nSample duplicate employees:")
print(
    duplicate_rows[
        [
            "Employee ID",
            "FirstName",
            "LastName",
            "StartDate",
            "ExitDate",
            "Title",
            "EmployeeStatus",
            "EmployeeType",
            "DepartmentType",
            "Training Date",
            "Training Program Name"
        ]
    ]
    .sort_values("Employee ID")
    .head(30)
)

DUPLICATE EMPLOYEE ID INVESTIGATION
Duplicate Employee IDs: 150
Rows belonging to duplicate IDs: 300

Rows per duplicated employee:
count
2    150
Name: count, dtype: int64

Sample duplicate employees:
      Employee ID FirstName  LastName  StartDate   ExitDate  \
612          1039     Nolan     Perez 2019-08-22 2022-03-25   
3061         1039     Nolan     Perez 2019-08-22 2022-03-25   
3012         1071    Yadira  Mcmillan 2019-03-28 2020-10-17   
644          1071    Yadira  Mcmillan 2019-03-28 2020-10-17   
685          1112   Celeste   Johnson 2022-07-10        NaT   
3102         1112   Celeste   Johnson 2022-07-10        NaT   
695          1122    Sierra    Macias 2021-11-16 2022-08-26   
3075         1122    Sierra    Macias 2021-11-16 2022-08-26   
3060         1145    Autumn   Barrera 2020-09-24        NaT   
718          1145    Autumn   Barrera 2020-09-24        NaT   
727          1154    Meadow     Avery 2021-09-27 2023-02-05   
3043         1154    Meadow     Avery 2021

In [24]:
# ============================================================
# STEP 9: CONFIRM TRUE DUPLICATES
# ============================================================

print("=" * 80)
print("TRUE DUPLICATE ANALYSIS")
print("=" * 80)

# Columns that describe the employee record
# Exclude Employee ID because we are checking records WITHIN each ID
check_columns = [
    col for col in messy_hr.columns
    if col != "Employee ID"
]

# Check whether each duplicated Employee ID has
# exactly one unique record across all other columns
duplicate_uniqueness = (
    duplicate_rows
    .groupby("Employee ID")[check_columns]
    .nunique(dropna=False)
)

# An employee is a true duplicate if every column has
# exactly one unique value
true_duplicate_ids = duplicate_uniqueness[
    (duplicate_uniqueness == 1).all(axis=1)
].index

print("Duplicate Employee IDs          :", len(duplicate_ids))
print("True duplicate Employee IDs     :", len(true_duplicate_ids))
print(
    "Potential non-identical duplicates:",
    len(duplicate_ids) - len(true_duplicate_ids)
)

print("\nRows that can safely be removed:")
print(len(true_duplicate_ids))

TRUE DUPLICATE ANALYSIS
Duplicate Employee IDs          : 150
True duplicate Employee IDs     : 150
Potential non-identical duplicates: 0

Rows that can safely be removed:
150


In [25]:
print("=" * 80)
print("REMOVING CONFIRMED TRUE DUPLICATES")
print("=" * 80)

before_rows = len(messy_hr)

# Remove duplicate Employee IDs
messy_hr_clean = messy_hr.drop_duplicates(
    subset=["Employee ID"],
    keep="first"
).copy()

after_rows = len(messy_hr_clean)

print(f"Rows before duplicate removal : {before_rows}")
print(f"Rows removed                   : {before_rows - after_rows}")
print(f"Rows after duplicate removal  : {after_rows}")

print("\nEmployee ID uniqueness:")
print(f"Unique IDs : {messy_hr_clean['Employee ID'].nunique()}")
print(f"Rows       : {len(messy_hr_clean)}")

REMOVING CONFIRMED TRUE DUPLICATES
Rows before duplicate removal : 3150
Rows removed                   : 150
Rows after duplicate removal  : 3000

Employee ID uniqueness:
Unique IDs : 3000
Rows       : 3000


In [26]:
print("=" * 80)
print("POST-DUPLICATE VALIDATION")
print("=" * 80)

duplicate_ids_after = (
    messy_hr_clean["Employee ID"]
    .value_counts()
    .loc[lambda x: x > 1]
)

print(f"Duplicate Employee IDs remaining : {len(duplicate_ids_after)}")

if len(duplicate_ids_after) == 0:
    print("STATUS: PASS - No duplicate Employee IDs remain.")
else:
    print("STATUS: FAIL - Duplicates still exist.")
    print(duplicate_ids_after)

POST-DUPLICATE VALIDATION
Duplicate Employee IDs remaining : 0
STATUS: PASS - No duplicate Employee IDs remain.


In [27]:
print("=" * 80)
print("CLEANED MESSY HR DATASET")
print("=" * 80)

print("Shape:", messy_hr_clean.shape)

print("\nUnique employees:")
print(messy_hr_clean["Employee ID"].nunique())

print("\nMissing values:")
print(messy_hr_clean.isna().sum().sum())

print("\nDuplicate Employee IDs:")
print(
    messy_hr_clean["Employee ID"]
    .duplicated()
    .sum()
)

CLEANED MESSY HR DATASET
Shape: (3000, 40)

Unique employees:
3000

Missing values:
3030

Duplicate Employee IDs:
0


In [28]:
print("=" * 80)
print("EMPLOYEE RECONCILIATION")
print("=" * 80)

clean_ids = set(clean_hr["Employee ID"])
messy_ids = set(messy_hr_clean["Employee ID"])

common_ids = clean_ids & messy_ids
messy_only_ids = messy_ids - clean_ids
clean_only_ids = clean_ids - messy_ids

print(f"Clean HR employees       : {len(clean_ids)}")
print(f"Cleaned Messy employees  : {len(messy_ids)}")
print(f"Common employees         : {len(common_ids)}")
print(f"Messy-only employees     : {len(messy_only_ids)}")
print(f"Clean-only employees     : {len(clean_only_ids)}")

EMPLOYEE RECONCILIATION
Clean HR employees       : 2845
Cleaned Messy employees  : 3000
Common employees         : 2845
Messy-only employees     : 155
Clean-only employees     : 0


In [33]:
print("=" * 80)
print("MISSING VALUE ANALYSIS")
print("=" * 80)

missing_summary = (
    messy_hr_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[missing_summary > 0]

print(missing_summary)

MISSING VALUE ANALYSIS
ExitDate                  1467
TerminationDescription    1467
DOB                         96
dtype: int64


In [34]:
print("\n" + "=" * 80)
print("MISSING VALUE PERCENTAGE")
print("=" * 80)

missing_percentage = (
    messy_hr_clean.isna().mean() * 100
).sort_values(ascending=False)

missing_percentage = missing_percentage[missing_percentage > 0]

for column, percentage in missing_percentage.items():
    print(f"{column:35} : {percentage:.2f}%")


MISSING VALUE PERCENTAGE
ExitDate                            : 48.90%
TerminationDescription              : 48.90%
DOB                                 : 3.20%


In [35]:
print("\n" + "=" * 80)
print("MISSING VALUES BY EMPLOYEE STATUS")
print("=" * 80)

missing_by_status = (
    messy_hr_clean
    .groupby("EmployeeStatus")
    .apply(lambda x: x.isna().sum(), include_groups=False)
)

print(missing_by_status)


MISSING VALUES BY EMPLOYEE STATUS
                        Unnamed: 0  FirstName  LastName  StartDate  ExitDate  \
EmployeeStatus                                                                 
Active                           0          0         0          0      1467   
Future Start                     0          0         0          0         0   
Leave of Absence                 0          0         0          0         0   
Terminated for Cause             0          0         0          0         0   
Voluntarily Terminated           0          0         0          0         0   

                        Title  Supervisor  ADEmail  BusinessUnit  \
EmployeeStatus                                                     
Active                      0           0        0             0   
Future Start                0           0        0             0   
Leave of Absence            0           0        0             0   
Terminated for Cause        0           0        0             0

In [36]:
print("\n" + "=" * 80)
print("MISSING VALUES BY EMPLOYEE TYPE")
print("=" * 80)

missing_by_type = (
    messy_hr_clean
    .groupby("EmployeeType")
    .apply(lambda x: x.isna().sum(), include_groups=False)
)

print(missing_by_type)


MISSING VALUES BY EMPLOYEE TYPE
              Unnamed: 0  FirstName  LastName  StartDate  ExitDate  Title  \
EmployeeType                                                                
Contract               0          0         0          0       494      0   
Full-Time              0          0         0          0       504      0   
Part-Time              0          0         0          0       469      0   

              Supervisor  ADEmail  BusinessUnit  EmployeeStatus  ...  \
EmployeeType                                                     ...   
Contract               0        0             0               0  ...   
Full-Time              0        0             0               0  ...   
Part-Time              0        0             0               0  ...   

              Work-Life Balance Score  Training Date  Training Program Name  \
EmployeeType                                                                  
Contract                            0              0          

In [37]:
print("=" * 80)
print("FINAL MISSING DATA FLAGS")
print("=" * 80)

messy_hr_clean["DOBMissing"] = messy_hr_clean["DOB"].isna()

print("Missing DOB:", messy_hr_clean["DOBMissing"].sum())
print("Available DOB:", messy_hr_clean["DOB"].notna().sum())

FINAL MISSING DATA FLAGS
Missing DOB: 96
Available DOB: 2904


In [38]:
print("=" * 80)
print("POST-CLEANING VALIDATION")
print("=" * 80)

print("Shape:", messy_hr_clean.shape)

print("\nUnique Employee IDs:",
      messy_hr_clean["Employee ID"].nunique())

print("\nDuplicate Employee IDs:",
      messy_hr_clean["Employee ID"].duplicated().sum())

print("\nTotal missing values:",
      messy_hr_clean.isna().sum().sum())

print("\nMissing values by column:")
print(
    messy_hr_clean.isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

POST-CLEANING VALIDATION
Shape: (3000, 40)

Unique Employee IDs: 3000

Duplicate Employee IDs: 0

Total missing values: 3030

Missing values by column:
ExitDate                  1467
TerminationDescription    1467
DOB                         96
Unnamed: 0                   0
StartDate                    0
Title                        0
LastName                     0
FirstName                    0
ADEmail                      0
Supervisor                   0
dtype: int64


In [39]:
print("=" * 80)
print("CREATING DATA QUALITY FLAGS")
print("=" * 80)

# Training happened before employee started
messy_hr_clean["TrainingBeforeStart"] = (
    messy_hr_clean["Training Date"] < messy_hr_clean["StartDate"]
)

# Survey happened before employee started
messy_hr_clean["SurveyBeforeStart"] = (
    messy_hr_clean["Survey Date"] < messy_hr_clean["StartDate"]
)

# DOB is missing
messy_hr_clean["DOBMissing"] = (
    messy_hr_clean["DOB"].isna()
)

print("TrainingBeforeStart:",
      messy_hr_clean["TrainingBeforeStart"].sum())

print("SurveyBeforeStart:",
      messy_hr_clean["SurveyBeforeStart"].sum())

print("DOBMissing:",
      messy_hr_clean["DOBMissing"].sum())

CREATING DATA QUALITY FLAGS
TrainingBeforeStart: 288
SurveyBeforeStart: 287
DOBMissing: 96


In [40]:
# ================================================================
# FINAL CLEAN DATASET
# ================================================================

print("=" * 80)
print("FINAL CLEAN DATASET")
print("=" * 80)

# Make a final copy
final_hr = messy_hr_clean.copy()

print("Shape:", final_hr.shape)
print("Unique employees:", final_hr["Employee ID"].nunique())
print("Duplicate IDs:", final_hr["Employee ID"].duplicated().sum())
print("Missing values:", final_hr.isna().sum().sum())

print("\nQuality Flags:")
print("TrainingBeforeStart:", final_hr["TrainingBeforeStart"].sum())
print("SurveyBeforeStart:", final_hr["SurveyBeforeStart"].sum())
print("DOBMissing:", final_hr["DOBMissing"].sum())

FINAL CLEAN DATASET
Shape: (3000, 42)
Unique employees: 3000
Duplicate IDs: 0
Missing values: 3030

Quality Flags:
TrainingBeforeStart: 288
SurveyBeforeStart: 287
DOBMissing: 96


In [41]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

final_hr.to_csv(
    PROCESSED_DIR / "hr_cleaned.csv",
    index=False
)

print("\nSaved:")
print(PROCESSED_DIR / "hr_cleaned.csv")


Saved:
..\data\processed\hr_cleaned.csv
